# 12 · Trained baselines 2/2: run the four baselines (design doc §8.8.16)

**How to run**: finish `11_trained_baselines_data.ipynb` first. Run the first cell to free GPU memory; pick a Colab server with an **A100 GPU** and click Run All; about 1.5 hours (AdaDetectGPT, which scores text by text, is the slowest). A rerun after a disconnect skips saved stages. Results go to `DATA_DIR/11_trained_baselines/scores_trained.csv`.

**Four baselines**: MAGE Longformer (off the shelf), the RoBERTa-large OpenAI detector (off the shelf), DivEye (official code, GPT-2 features, XGBoost) and AdaDetectGPT (official witness function and statistic; Falcon-7B-Instruct scorer, Falcon-7B reference). The two trained ones are fitted on all three generators and, separately, leaving one generator out.


In [ ]:
# 1. Check the GPU. Two 7B models need about 30 GB free.
import torch
assert torch.cuda.is_available(), "No GPU: select an A100 runtime."
free = torch.cuda.mem_get_info()[0] / 2**30
print(torch.cuda.get_device_name(0), f"{free:.0f} GB free")
assert free > 30, "Not enough free GPU memory; restart the runtime."


In [ ]:
# 2. Google Drive, packages, official code (commits recorded)
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.environ["HF_HOME"] = "/content/hf"
%pip install -q -U transformers huggingface-hub accelerate xgboost scikit-learn "clean-text[gpl]" sacremoses regex
import sys, json, gc, hashlib, platform
import numpy as np, pandas as pd, transformers
from datetime import datetime, timezone
commits = {}
for name, url in (("MAGE", "https://github.com/yafuly/MAGE"), ("diveye", "https://github.com/IBM/diveye"), ("AdaDetectGPT", "https://github.com/Mamba413/AdaDetectGPT")):
    if not os.path.exists(f"/content/{name}"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, f"/content/{name}"], check=True)
    commits[name] = subprocess.run(["git", "-C", f"/content/{name}", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print(commits)
T = pd.read_csv("11_trained_baselines/texts.csv"); info = json.load(open("11_trained_baselines/data_info.json"))
assert hashlib.sha256(open("11_trained_baselines/texts.csv", "rb").read()).hexdigest() == info["sha256"], "texts.csv does not match the hash in data_info.json"
T["text"] = T.text.astype(str); print(T.groupby(["set", "who"]).size().to_string())
META = ["set", "id", "domain", "model", "decoding", "who"]
GENS3 = ["Qwen2.5-3B", "phi-2", "SmolLM2-1.7B"]


In [ ]:
# 3. Off-the-shelf detectors: MAGE Longformer (official preprocessing; score = probability of class 0, "machine") and the RoBERTa-large OpenAI detector (score = probability of index 0, "fake")
from transformers import AutoTokenizer, AutoModelForSequenceClassification
OUT1 = "11_trained_baselines/scores_offtheshelf.csv"
if os.path.exists(OUT1):
    print("Already done, skipping:", OUT1)
else:
    sys.path.insert(0, "/content/MAGE"); from deployment import preprocess
    from transformers import AutoConfig, PretrainedConfig
    def load_config(model_name):   # yaful/MAGE's config.json stores labels as numbers {"0": 0, "1": 1}; transformers 5 accepts only strings, so convert to {"0": "0", "1": "1"} first
        d, _ = PretrainedConfig.get_config_dict(model_name)
        if d.get("id2label"):
            d["id2label"] = {k: str(v) for k, v in d["id2label"].items()}
        return AutoConfig.for_model(d.pop("model_type"), **d)
    @torch.no_grad()
    def clf_scores(model_name, texts, bs, prep=None):
        cfg = load_config(model_name)
        tok = AutoTokenizer.from_pretrained(model_name, config=cfg); m = AutoModelForSequenceClassification.from_pretrained(model_name, config=cfg).cuda().eval()
        out, t0 = [], time.time()
        for s in range(0, len(texts), bs):
            chunk = [prep(t) if prep else t for t in texts[s:s + bs]]
            enc = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            out.extend(m(**enc).logits.float().softmax(-1)[:, 0].cpu().tolist())
            if (s // bs) % 100 == 0: print(f"  {model_name}: {s}/{len(texts)} {(time.time() - t0) / 60:.1f} min", flush=True)
        del m; gc.collect(); torch.cuda.empty_cache(); return out
    texts = T.text.tolist()
    R1 = T[META].copy()
    R1["mage_longformer"] = clf_scores("yaful/MAGE", texts, 16, prep=preprocess)
    R1["roberta_openai"] = clf_scores("openai-community/roberta-large-openai-detector", texts, 32)
    R1.to_csv(OUT1, index=False); print("saved", OUT1)


In [ ]:
# 4. DivEye official features (GPT-2, diveye_utils.DivEyeUtils) + official XGBoost settings; trained on all three generators and leaving one out
FEAT = "11_trained_baselines/diveye_features.csv"; OUT2 = "11_trained_baselines/scores_diveye.csv"
if os.path.exists(OUT2):
    print("Already done, skipping:", OUT2)
else:
    sys.path.insert(0, "/content/diveye")
    from diveye_utils import DivEyeUtils
    from transformers import AutoModelForCausalLM
    import xgboost as xgb
    if os.path.exists(FEAT):
        F = pd.read_csv(FEAT)
    else:
        g2 = AutoModelForCausalLM.from_pretrained("gpt2").cuda().eval(); g2tok = AutoTokenizer.from_pretrained("gpt2")
        utils = DivEyeUtils(g2, g2tok); feats, t0 = [], time.time()
        NAN = [float("nan")] * 10
        for i, t in enumerate(T.text.tolist()):
            if len(g2tok.encode(t, truncation=True, max_length=1024)) < 11:   # skip texts that are too short (one in M4GT); the official code recurses forever on the error
                feats.append(NAN); continue
            try:
                feats.append(utils.diveye_compute(t))
            except Exception as e:
                print("  DivEye features failed, filling NaN:", i, type(e).__name__); feats.append(NAN)
            if i % 2000 == 0: print(f"  DivEye features {i}/{len(T)} {(time.time() - t0) / 60:.1f} min", flush=True)
        F = pd.concat([T[META].reset_index(drop=True), pd.DataFrame(feats, columns=[f"f{i}" for i in range(10)])], axis=1)
        F.to_csv(FEAT, index=False); del g2; gc.collect(); torch.cuda.empty_cache()
    fcols = [f"f{i}" for i in range(10)]
    def fit(train):   # hyperparameters of the official diveye.py; label 1 = AI
        y = (train.who == "ai").astype(int).values
        clf = xgb.XGBClassifier(random_state=42, scale_pos_weight=(len(y) - y.sum()) / y.sum(), max_depth=12, n_estimators=200, colsample_bytree=0.8, subsample=0.7, min_child_weight=5, gamma=1.0)
        clf.fit(train[fcols].values, y); return clf
    tr = F[F.set == "train"]; test = F[F.set != "train"].copy()
    R2 = test[META].copy()
    R2["diveye_all3"] = fit(tr).predict_proba(test[fcols].values)[:, 1]
    for g in GENS3:   # leave one generator out: applied only to that generator's AI texts and the humans of set main1
        clf = fit(tr[tr.model != g]); mask = (test.set == "main1") & ((test.model == g) | (test.who == "human"))
        R2[f"diveye_logo_{g}"] = np.where(mask, clf.predict_proba(test[fcols].values)[:, 1], np.nan)
    R2.to_csv(OUT2, index=False); print("saved", OUT2)


In [ ]:
# 5. AdaDetectGPT: official BSplineTwoSample witness function (trained) and get_classification_stat (statistic, copied verbatim); Falcon dual models
OUT3 = "11_trained_baselines/scores_ada_v2.csv"   # in v1 the witness function degenerated to a constant (1,000 humans vs 3,600 AI, unpaired); v2 uses paired data
if os.path.exists(OUT3):
    print("Already done, skipping:", OUT3)
else:
    sys.path.insert(0, "/content/AdaDetectGPT/scripts")
    from nuisance_func import BSplineTwoSample
    from torch import nn
    def get_classification_stat(logits_ref, logits_score, labels, w_func, shift_value=None):   # detect_gpt_ada.py, copied verbatim
        assert logits_ref.shape[0] == 1
        assert logits_score.shape[0] == 1
        assert labels.shape[0] == 1
        if logits_ref.size(-1) != logits_score.size(-1):
            vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
            logits_ref = logits_ref[:, :, :vocab_size]
            logits_score = logits_score[:, :, :vocab_size]
        labels = labels.unsqueeze(-1) if labels.ndim == logits_score.ndim - 1 else labels
        lprobs_score = w_func(torch.log_softmax(logits_score, dim=-1))
        probs_ref = torch.softmax(logits_ref, dim=-1)
        log_likelihood = lprobs_score.gather(dim=-1, index=labels).squeeze(-1)
        mean_ref = (probs_ref * lprobs_score).sum(dim=-1)
        var_ref = (probs_ref * torch.square(lprobs_score)).sum(dim=-1) - torch.square(mean_ref)
        L = torch.tensor(var_ref.shape[0])
        stat = (log_likelihood.sum(dim=-1) - mean_ref.sum(dim=-1) - shift_value * L) / var_ref.sum(dim=-1).sqrt()
        stat = stat.mean()
        return stat.item()
    from transformers import AutoModelForCausalLM
    for _n in ("p_model", "q_model"): globals().pop(_n, None)
    gc.collect(); torch.cuda.empty_cache()
    SCORER, OBSERVER, MAX_TOK, BS = "tiiuae/falcon-7b-instruct", "tiiuae/falcon-7b", 512, 8
    tok = AutoTokenizer.from_pretrained(SCORER); tok.padding_side = "right"
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token   # Falcon has no pad token and padding would fail; the official model.py also uses eos as pad
    pad = tok.pad_token_id
    p_model = AutoModelForCausalLM.from_pretrained(SCORER, dtype=torch.bfloat16).cuda().eval()   # scorer
    q_model = AutoModelForCausalLM.from_pretrained(OBSERVER, dtype=torch.bfloat16).cuda().eval() # reference (sampling)
    print("loaded", f"{torch.cuda.memory_allocated() / 2**30:.1f} GB")
    CONFIG = {"start": -32, "end": 0, "n_bases": 7, "spline_order": 2, "intercept": 1}   # official defaults
    class A: device = "cuda"
    # training: the official fit = get_zij(human) + get_zij(machine) + compute_beta_hat. Here get_zij runs once per training text and is combined per variant, which equals calling fit for each variant.
    BETA = "11_trained_baselines/ada_beta_v2.json"
    if os.path.exists(BETA):   # reuse the saved training result, saving about 15 minutes
        W = {}
        for k, v in json.load(open(BETA)).items():
            W[k] = BSplineTwoSample(CONFIG, "cuda"); W[k].beta_hat = torch.tensor(v, dtype=torch.float32, device="cuda")
        print("Loaded witness functions:", list(W))
    else:
        tr = T[T.set == "train"].reset_index(drop=True)
        w0 = BSplineTwoSample(CONFIG, "cuda")
        tok_list = [tok(x, return_tensors="pt", padding=True, return_token_type_ids=False).to("cuda") for x in tr.text.tolist()]
        z = w0.get_zij(tok_list, p_model, A()); del tok_list
        W = {}
        # the official data format is paired, {"original": [...], "sampled": [...]}, with equal human and machine counts; compute_beta_hat sums for delta,
        # so with unequal counts the constant term absorbs all weight (the v1 error). AI texts are therefore subsampled per (generator x temperature) to the human count, seed 0.
        hum_idx = np.where(tr.who == "human")[0]; rs = np.random.default_rng(0)
        def balanced(mask):
            pool = tr[mask]; groups = list(pool.groupby(["model", "decoding"]).groups.values())
            per = len(hum_idx) // len(groups); pick = np.concatenate([rs.choice(np.asarray(g), per, replace=False) for g in groups])
            rest = rs.choice(np.setdiff1d(pool.index.values, pick), len(hum_idx) - len(pick), replace=False)
            return np.concatenate([pick, rest])
        for name, mask in [("all3", tr.who == "ai")] + [(f"logo_{g}", (tr.who == "ai") & (tr.model != g)) for g in GENS3]:
            w = BSplineTwoSample(CONFIG, "cuda"); ai_idx = balanced(mask); assert len(ai_idx) == len(hum_idx)
            w.beta_hat = w.compute_beta_hat([z[i] for i in hum_idx], [z[i] for i in ai_idx], False)
            assert w.beta_hat[0].abs() < 0.9, f"{name}: the witness function degenerated to a constant again {w.beta_hat.tolist()}"
            W[name] = w; print(name, "beta_hat:", torch.round(w.beta_hat, decimals=3).tolist())
        json.dump({k: v.beta_hat.detach().cpu().tolist() for k, v in W.items()}, open("11_trained_baselines/ada_beta_v2.json", "w"), indent=2)
    W["identity"] = nn.Identity(); zero = torch.zeros(1, device="cuda")
    # scoring: batched forward passes of both models (right padding), then the official statistic per text; LOGO variants score only that generator's AI texts and the humans of set main1
    test = T[T.set != "train"].reset_index(drop=True)
    def variants(r):
        v = ["identity", "all3"]
        if r.set == "main1": v += [f"logo_{g}" for g in GENS3 if r.who == "human" or r.model == g]
        return v
    enc = [(i, tok(t, truncation=True, max_length=MAX_TOK)["input_ids"]) for i, t in enumerate(test.text.tolist())]
    enc = [e for e in enc if len(e[1]) >= 11]; enc.sort(key=lambda e: len(e[1]))
    PART = OUT3 + ".part.pkl"   # progress file, saved every 50 batches; a rerun resumes
    out = pd.read_pickle(PART) if os.path.exists(PART) else {}
    enc = [e for e in enc if e[0] not in out]; print("Scored", len(out), "texts;", len(enc), "left")
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, len(enc), BS):
            chunk = enc[s:s + BS]; L = max(len(e[1]) for e in chunk)
            x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
            for i, (_, ids) in enumerate(chunk):
                x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
            x, att = x.cuda(), att.cuda()
            lg_p = p_model(input_ids=x, attention_mask=att).logits; lg_q = q_model(input_ids=x, attention_mask=att).logits
            for i, (row, ids) in enumerate(chunk):
                n = len(ids); r = test.iloc[row]
                ls, lr = lg_p[i:i + 1, :n - 1].float(), lg_q[i:i + 1, :n - 1].float(); lab = torch.tensor([ids[1:]], device="cuda")
                out[row] = {f"ada_{v}": get_classification_stat(lr, ls, lab, W[v], zero) for v in variants(r)}
            if (s // BS) % 50 == 0: pd.to_pickle(out, PART)
            if (s // BS) % 100 == 0: print(f"  AdaDetectGPT {s}/{len(enc)} {(time.time() - t0) / 60:.1f} min", flush=True)
    R3 = pd.concat([test[META], pd.DataFrame.from_dict(out, orient="index").reindex(range(len(test)))], axis=1)
    R3.to_csv(OUT3, index=False); os.path.exists(PART) and os.remove(PART); print("saved", OUT3, f"{(time.time() - t0) / 60:.1f} min")
    del p_model, q_model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 6. Merge into scores_trained.csv and record run information
R1 = pd.read_csv(OUT1); R1 = R1[R1.set != "train"]
R = R1.merge(pd.read_csv(OUT2), on=META, how="left").merge(pd.read_csv(OUT3), on=META, how="left")
assert len(R) == (T.set != "train").sum(), (len(R), (T.set != "train").sum())
R.to_csv("11_trained_baselines/scores_trained.csv", index=False)
print(R.drop(columns=META).notna().sum().to_string())
run_info = dict(completed_at_utc=datetime.now(timezone.utc).isoformat(), commits=commits, texts_sha256=info["sha256"], scorer="tiiuae/falcon-7b-instruct", observer="tiiuae/falcon-7b",
                diveye_feature_model="gpt2", ada_config={"start": -32, "end": 0, "n_bases": 7, "spline_order": 2, "intercept": 1}, python=sys.version, torch=torch.__version__,
                transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(run_info, open("11_trained_baselines/run_info.json", "w"), ensure_ascii=False, indent=2); print(json.dumps(run_info, ensure_ascii=False, indent=2))


In [ ]:
%cd /content
drive.flush_and_unmount(); print("Done: results are in DATA_DIR/11_trained_baselines/scores_trained.csv")
